# Notebook 23 — Disparity Map
### Heterogeneous Treatment Effects in Mortgage Lending

**Author:** Rajveer Singh Pall  
**Institution:** Gyan Ganga Institute of Technology and Sciences

---

## What this notebook does

Builds the flagship visualisation of the paper: a personalised disparity map
showing how the racial approval penalty varies across income, LTV, and lender type.

This is the figure that answers the core question in one image:
who bears the largest share of the racial approval gap, and where?

**Three visualisations:**
1. Income x LTV heatmap — the 2D disparity map (main figure)
2. Income x AUS type heatmap — discretion channel map
3. Year x income heatmap — temporal evolution of heterogeneity

**INPUT:** `data/cate_estimates.parquet` (from NB21)

**OUTPUTS:**
- `outputs/figures/nb23_disparity_map_income_ltv.png`
- `outputs/figures/nb23_disparity_map_income_aus.png`
- `outputs/figures/nb23_disparity_map_temporal.png`
- `outputs/tables/nb23_disparity_cells.csv`

**RUNTIME:** ~5-10 minutes

In [ ]:
# CELL 1 - IMPORTS
import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

BASE_DIR    = Path('D:/CATE-HMDA-Heterogeneous-Effects')
DATA_DIR    = BASE_DIR / 'data'
TABLES_DIR  = BASE_DIR / 'outputs' / 'tables'
FIGURES_DIR = BASE_DIR / 'outputs' / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi':        150,
    'font.family':       'serif',
    'font.size':         11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

# Verify input
fp = DATA_DIR / 'cate_estimates.parquet'
assert fp.exists(), 'Run NB21 first — cate_estimates.parquet not found'

n = pl.scan_parquet(str(fp)).select(pl.len()).collect().item()
cols = pl.scan_parquet(str(fp)).columns
print(f'cate_estimates.parquet: {n:,} rows')
print(f'Columns: {cols}')

In [ ]:
# CELL 2 - LOAD CATE ESTIMATES
print('='*70)
print('LOADING CATE ESTIMATES')
print('='*70)

df = pl.read_parquet(str(DATA_DIR / 'cate_estimates.parquet')).to_pandas()

print(f'Rows          : {len(df):,}')
print(f'CATE mean     : {df["cate_pp"].mean():.2f} pp')
print(f'CATE std      : {df["cate_pp"].std():.2f} pp')

# Create income quintile labels
df['income_q'] = pd.qcut(
    df['income'], q=5,
    labels=['Q1\n<$60K', 'Q2\n$60-90K', 'Q3\n$90-125K', 'Q4\n$125-180K', 'Q5\n>$180K']
)

# Create LTV bins
df['ltv_bin'] = pd.cut(
    df['ltv'], bins=[0, 60, 70, 75, 80, 90, 100, 200],
    labels=['<=60%', '60-70%', '70-75%', '75-80%', '80-90%', '90-100%', '>100%']
)

# Check AUS column availability
has_aus = 'aus_automated' in df.columns
has_purpose = 'purpose_purchase' in df.columns
print(f'AUS column    : {"available" if has_aus else "not in cate_df"}')
print(f'Purpose column: {"available" if has_purpose else "not in cate_df"}')

# AUS type label
if has_aus:
    df['aus_label'] = df['aus_automated'].map({1: 'Automated\n(AUS)', 0: 'Manual/\nexempt'})

print('Data prepared for heatmaps')

In [ ]:
# CELL 3 - FIGURE 1: INCOME x LTV DISPARITY MAP (MAIN FIGURE)
#
# This is the flagship figure of the paper.
# Each cell shows the mean CATE for applicants in that income-LTV cell.
# Color: red = larger penalty, blue = smaller penalty.
# Cell text: mean CATE in pp.
# Cell size (bubble): number of applicants in that cell.
print('Generating Income x LTV disparity map...')

# Pivot table: rows = LTV bins, cols = income quintiles
pivot = df.groupby(['ltv_bin', 'income_q'], observed=True)['cate_pp'].agg(
    ['mean', 'count', 'std']
).reset_index()
pivot.columns = ['ltv_bin', 'income_q', 'mean_cate', 'n', 'std_cate']
pivot['se'] = pivot['std_cate'] / np.sqrt(pivot['n'])

# Create pivot matrix for heatmap
heatmap_data = pivot.pivot(index='ltv_bin', columns='income_q', values='mean_cate')
heatmap_n    = pivot.pivot(index='ltv_bin', columns='income_q', values='n')

# LTV order (low to high)
ltv_order = ['<=60%', '60-70%', '70-75%', '75-80%', '80-90%', '90-100%', '>100%']
ltv_order = [l for l in ltv_order if l in heatmap_data.index]
heatmap_data = heatmap_data.reindex(ltv_order)
heatmap_n    = heatmap_n.reindex(ltv_order)

fig, ax = plt.subplots(figsize=(12, 7))

# Custom diverging colormap centered at the overall mean
vmin = heatmap_data.values.min()
vmax = heatmap_data.values.max()
overall_mean = df['cate_pp'].mean()

# Use RdBu_r: red = more negative (larger penalty), blue = less negative
cmap = plt.cm.RdBu_r
norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=overall_mean, vmax=vmax)

im = ax.imshow(
    heatmap_data.values,
    cmap=cmap, norm=norm,
    aspect='auto'
)

# Add cell text
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        val = heatmap_data.values[i, j]
        n_cell = heatmap_n.values[i, j]
        if not np.isnan(val):
            # Main CATE value
            ax.text(j, i, f'{val:.1f} pp',
                    ha='center', va='center',
                    fontsize=10, fontweight='bold',
                    color='white' if abs(val - overall_mean) > 3 else 'black')
            # Sample size below
            ax.text(j, i + 0.32, f'n={n_cell:,}',
                    ha='center', va='center',
                    fontsize=7,
                    color='white' if abs(val - overall_mean) > 3 else 'black')

# Axes labels
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, fontsize=10)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index, fontsize=10)
ax.set_xlabel('Income quintile', fontsize=12, labelpad=10)
ax.set_ylabel('Loan-to-value ratio', fontsize=12, labelpad=10)

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Mean CATE (pp)\nMore red = larger penalty', fontsize=10)

# PMI threshold line
try:
    pmi_row = heatmap_data.index.tolist().index('80-90%') - 0.5
    ax.axhline(pmi_row, color='black', linewidth=2.5, linestyle='--')
    ax.text(len(heatmap_data.columns) - 0.45, pmi_row - 0.15,
            '80% LTV (PMI boundary)', fontsize=9, color='black',
            ha='right', style='italic')
except (ValueError, IndexError):
    pass

ax.set_title(
    'Figure NB23-1: Personalised Disparity Map\n'
    'Mean CATE by Income Quintile x Loan-to-Value Ratio\n'
    'Racial approval penalty (pp) — more red = larger penalty',
    fontsize=12, pad=15
)

plt.tight_layout()
out = FIGURES_DIR / 'nb23_disparity_map_income_ltv.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out.name}')

# Print the matrix
print('\nMean CATE by Income x LTV (pp):')
print(heatmap_data.round(2).to_string())

In [ ]:
# CELL 4 - FIGURE 2: INCOME x AUS TYPE MAP
#
# Shows how the AUS channel interacts with income.
# Key question: does automated underwriting protect all income groups equally,
# or does it protect high-income Black applicants more?
print('Generating Income x AUS disparity map...')

if has_aus:
    pivot_aus = df.groupby(['aus_label', 'income_q'], observed=True)['cate_pp'].agg(
        ['mean', 'count']
    ).reset_index()
    pivot_aus.columns = ['aus_label', 'income_q', 'mean_cate', 'n']

    heatmap_aus  = pivot_aus.pivot(index='aus_label', columns='income_q', values='mean_cate')
    heatmap_aus_n = pivot_aus.pivot(index='aus_label', columns='income_q', values='n')

    fig, ax = plt.subplots(figsize=(12, 4))

    vmin_a = heatmap_aus.values.min()
    vmax_a = heatmap_aus.values.max()
    norm_a = mcolors.TwoSlopeNorm(vmin=vmin_a, vcenter=overall_mean, vmax=vmax_a)

    im_a = ax.imshow(heatmap_aus.values, cmap=cmap, norm=norm_a, aspect='auto')

    for i in range(len(heatmap_aus.index)):
        for j in range(len(heatmap_aus.columns)):
            val   = heatmap_aus.values[i, j]
            n_c   = heatmap_aus_n.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.1f} pp',
                        ha='center', va='center',
                        fontsize=11, fontweight='bold',
                        color='white' if abs(val - overall_mean) > 3 else 'black')
                ax.text(j, i + 0.32, f'n={n_c:,}',
                        ha='center', va='center', fontsize=8,
                        color='white' if abs(val - overall_mean) > 3 else 'black')

    ax.set_xticks(range(len(heatmap_aus.columns)))
    ax.set_xticklabels(heatmap_aus.columns, fontsize=10)
    ax.set_yticks(range(len(heatmap_aus.index)))
    ax.set_yticklabels(heatmap_aus.index, fontsize=10)
    ax.set_xlabel('Income quintile', fontsize=12, labelpad=10)
    ax.set_ylabel('Underwriting type', fontsize=12, labelpad=10)

    cbar_a = plt.colorbar(im_a, ax=ax, fraction=0.046, pad=0.04)
    cbar_a.set_label('Mean CATE (pp)', fontsize=10)

    ax.set_title(
        'Figure NB23-2: Disparity Map — Income x Underwriting Type\n'
        'AUS type is the dominant moderator of the racial approval penalty',
        fontsize=12, pad=12
    )

    plt.tight_layout()
    out = FIGURES_DIR / 'nb23_disparity_map_income_aus.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out.name}')
    print('\nMean CATE by AUS x Income (pp):')
    print(heatmap_aus.round(2).to_string())
else:
    print('aus_automated not in cate_df — skipping AUS heatmap')
    print('This is fine. AUS analysis was done in NB21 subgroup table.')
    out = FIGURES_DIR / 'nb23_disparity_map_income_aus.png'
    # Create placeholder
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.text(0.5, 0.5, 'AUS column not available in cate_estimates.parquet\nSee NB21 subgroup table for AUS analysis',
            ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.set_title('NB23 AUS Map — see NB21 for AUS subgroup results')
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# CELL 5 - FIGURE 3: YEAR x INCOME TEMPORAL MAP
#
# Shows how the income gradient in the CATE evolved over time.
# Key question: did the 2022 tightening affect all income groups equally?
print('Generating Year x Income temporal map...')

pivot_yr = df.groupby(['year', 'income_q'], observed=True)['cate_pp'].agg(
    ['mean', 'count']
).reset_index()
pivot_yr.columns = ['year', 'income_q', 'mean_cate', 'n']

heatmap_yr   = pivot_yr.pivot(index='year', columns='income_q', values='mean_cate')
heatmap_yr_n = pivot_yr.pivot(index='year', columns='income_q', values='n')

fig, ax = plt.subplots(figsize=(12, 5))

vmin_y = heatmap_yr.values.min()
vmax_y = heatmap_yr.values.max()
norm_y = mcolors.TwoSlopeNorm(vmin=vmin_y, vcenter=overall_mean, vmax=vmax_y)

im_y = ax.imshow(heatmap_yr.values, cmap=cmap, norm=norm_y, aspect='auto')

for i in range(len(heatmap_yr.index)):
    for j in range(len(heatmap_yr.columns)):
        val = heatmap_yr.values[i, j]
        n_c = heatmap_yr_n.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.1f}',
                    ha='center', va='center',
                    fontsize=11, fontweight='bold',
                    color='white' if abs(val - overall_mean) > 3 else 'black')
            ax.text(j, i + 0.32, f'n={n_c:,}',
                    ha='center', va='center', fontsize=8,
                    color='white' if abs(val - overall_mean) > 3 else 'black')

ax.set_xticks(range(len(heatmap_yr.columns)))
ax.set_xticklabels(heatmap_yr.columns, fontsize=10)
ax.set_yticks(range(len(heatmap_yr.index)))
ax.set_yticklabels([str(int(y)) for y in heatmap_yr.index], fontsize=10)
ax.set_xlabel('Income quintile', fontsize=12, labelpad=10)
ax.set_ylabel('Year', fontsize=12, labelpad=10)

# Mark 2022 tightening
try:
    yr_list  = list(heatmap_yr.index)
    yr_2022  = yr_list.index(2022)
    ax.axhline(yr_2022 - 0.5, color='black', linewidth=2, linestyle='--')
    ax.text(len(heatmap_yr.columns) - 0.45, yr_2022 - 0.65,
            'Fed tightening begins (2022)', fontsize=9,
            ha='right', style='italic', color='black')
except (ValueError, IndexError):
    pass

cbar_y = plt.colorbar(im_y, ax=ax, fraction=0.046, pad=0.04)
cbar_y.set_label('Mean CATE (pp)', fontsize=10)

ax.set_title(
    'Figure NB23-3: Temporal Disparity Map — Year x Income Quintile\n'
    'How did the racial approval penalty evolve across income groups?',
    fontsize=12, pad=12
)

plt.tight_layout()
out = FIGURES_DIR / 'nb23_disparity_map_temporal.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out.name}')
print('\nMean CATE by Year x Income (pp):')
print(heatmap_yr.round(2).to_string())

In [ ]:
# CELL 6 - SAVE DISPARITY CELLS TABLE
print('='*70)
print('SAVING DISPARITY CELLS TABLE')
print('='*70)

# Full cell-level summary for paper appendix
cells = df.groupby(
    ['income_q', 'ltv_bin'], observed=True
)['cate_pp'].agg(['mean', 'median', 'std', 'count']).reset_index()
cells.columns = ['income_q', 'ltv_bin', 'mean_cate', 'median_cate', 'std_cate', 'n']
cells['se']     = cells['std_cate'] / np.sqrt(cells['n'])
cells['ci_lo']  = cells['mean_cate'] - 1.96 * cells['se']
cells['ci_hi']  = cells['mean_cate'] + 1.96 * cells['se']
cells['pct_penalised'] = df.groupby(
    ['income_q', 'ltv_bin'], observed=True
)['cate_pp'].apply(lambda x: 100*(x < 0).mean()).values

cells = cells.sort_values(['income_q', 'ltv_bin'])
cells.to_csv(TABLES_DIR / 'nb23_disparity_cells.csv', index=False)
print('Saved: nb23_disparity_cells.csv')

# Print most and least affected cells
print('\nTop 5 most penalised cells (income x LTV):')
print(cells.nsmallest(5, 'mean_cate')[['income_q', 'ltv_bin', 'mean_cate', 'n']].to_string(index=False))
print('\nTop 5 least penalised cells:')
print(cells.nlargest(5, 'mean_cate')[['income_q', 'ltv_bin', 'mean_cate', 'n']].to_string(index=False))

In [ ]:
# CELL 7 - VERIFICATION
print('='*70)
print('VERIFICATION')
print('='*70)

# 1. Output files exist
for f in ['nb23_disparity_map_income_ltv.png',
          'nb23_disparity_map_income_aus.png',
          'nb23_disparity_map_temporal.png']:
    assert (FIGURES_DIR / f).exists(), f'Missing: {f}'
assert (TABLES_DIR / 'nb23_disparity_cells.csv').exists()
print('1. All output files present OK')

# 2. Heatmap has data in all cells
n_cells   = heatmap_data.notna().sum().sum()
tot_cells = heatmap_data.shape[0] * heatmap_data.shape[1]
print(f'2. Income x LTV heatmap: {n_cells}/{tot_cells} cells populated')

# 3. Most penalised cell
min_idx   = cells.nsmallest(1, 'mean_cate').iloc[0]
max_idx   = cells.nlargest(1, 'mean_cate').iloc[0]
print(f'3. Most penalised cell  : Income {min_idx["income_q"]}, LTV {min_idx["ltv_bin"]} = {min_idx["mean_cate"]:.2f} pp')
print(f'   Least penalised cell : Income {max_idx["income_q"]}, LTV {max_idx["ltv_bin"]} = {max_idx["mean_cate"]:.2f} pp')
print(f'   Range: {min_idx["mean_cate"]:.2f} to {max_idx["mean_cate"]:.2f} pp')

# 4. Temporal map shows tightening effect
pre_mean  = heatmap_yr.loc[heatmap_yr.index <= 2021].mean().mean()
post_mean = heatmap_yr.loc[heatmap_yr.index >= 2022].mean().mean()
print(f'4. Pre-2022 mean CATE  : {pre_mean:.2f} pp')
print(f'   Post-2022 mean CATE : {post_mean:.2f} pp')
print(f'   Tightening effect   : {post_mean - pre_mean:.2f} pp')

print('\n' + '='*70)
print('ALL CHECKS PASSED')
print(f'Income x LTV range    : {heatmap_data.values.min():.1f} to {heatmap_data.values.max():.1f} pp')
print(f'NB23 complete -> proceed to NB24_subgroup_rdd.ipynb')
print('='*70)